# RASA — Data Audit
Notebook ini memeriksa manifest, distribusi katalog, temporal split, dan quality gates. Seluruh data bersifat sintetis.

In [ ]:
from pathlib import Path
import json, sys
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
from rasa.data.quality import run_data_quality_checks
from rasa.data.schema import load_processed_dataset

In [ ]:
manifest = json.loads((ROOT / 'data/raw/dataset_manifest.json').read_text())
bundle = load_processed_dataset()
pd.Series({
    'version': manifest['dataset_version'], 'seed': manifest['seed'],
    'products': len(bundle.catalog), 'users': len(bundle.users),
    'train_events': len(bundle.train), 'validation_events': len(bundle.validation),
    'test_events': len(bundle.test),
})

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
bundle.catalog.groupby('category_name').size().sort_values().plot.barh(ax=axes[0], title='Produk per kategori')
bundle.catalog.query("availability_status == 'available'").groupby('category_name').price.median().sort_values().plot.barh(ax=axes[1], title='Median harga tersedia')
plt.tight_layout()

In [ ]:
report = run_data_quality_checks(bundle)
pd.DataFrame(report['checks'])[['check', 'status', 'severity']]
assert report['summary']['passed_all_critical']